In [11]:
from pathlib import Path
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import ipywidgets as widgets

from IPython.display import display, HTML, clear_output

In [12]:
# ============================================================
# DBSCAN OUTPUT ROOT
# ============================================================

DBSCAN_OUT_ROOT = Path("/media/spell/Spell-lab/Lidar/D.DBSCAN Dataset")


# ============================================================
# PILIH FILE UNTUK DIVISUALISASIKAN
# ============================================================
# Pilihan dataset: "development" atau "testing"

DATASET_MODE = "development"

ACTIVITY = "Bungkuk"
SUBJECT = "Teguh"
FILE_ID = 9

# Khusus testing
ROOM = "Controlled Room"


# ============================================================
# BUILD FILE PATH
# ============================================================

if DATASET_MODE == "development":
    CHECK_FILE_PATH = (
        DBSCAN_OUT_ROOT
        / "Dataset Development"
        / ACTIVITY
        / SUBJECT
        / f"{FILE_ID}.csv"
    )

elif DATASET_MODE == "testing":
    CHECK_FILE_PATH = (
        DBSCAN_OUT_ROOT
        / "Dataset Testing"
        / ROOM
        / ACTIVITY
        / SUBJECT
        / f"{FILE_ID}.csv"
    )

else:
    raise ValueError("DATASET_MODE harus 'development' atau 'testing'.")


print("===== DBSCAN VISUALIZATION CONFIG =====")
print(f"Dataset mode : {DATASET_MODE}")
print(f"File path    : {CHECK_FILE_PATH}")
print(f"File exists  : {CHECK_FILE_PATH.exists()}")

===== DBSCAN VISUALIZATION CONFIG =====
Dataset mode : development
File path    : /media/spell/Spell-lab/Lidar/D.DBSCAN Dataset/Dataset Development/Bungkuk/Teguh/9.csv
File exists  : True


In [13]:
REQUIRED_COLUMNS = [
    "frame_id",
    "Timestamp",
    "X",
    "Y",
    "Z",
    "X_corr",
    "Y_corr",
    "Z_corr",
    "Z_level",
    "Reflectivity",
]

if not CHECK_FILE_PATH.exists():
    raise FileNotFoundError(f"File tidak ditemukan: {CHECK_FILE_PATH}")

df = pd.read_csv(CHECK_FILE_PATH)

missing_cols = [col for col in REQUIRED_COLUMNS if col not in df.columns]
if missing_cols:
    raise ValueError(f"Kolom berikut tidak ditemukan: {missing_cols}")

df = df[REQUIRED_COLUMNS].copy()

for col in ["frame_id", "X_corr", "Y_corr", "Z_corr", "Z_level"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["frame_id", "X_corr", "Y_corr", "Z_level"]).copy()
df["frame_id"] = df["frame_id"].astype(int)

frame_ids = sorted(df["frame_id"].unique().tolist())

print("===== FILE LOADED =====")
print(f"Shape       : {df.shape}")
print(f"Total frame : {len(frame_ids)}")
print(f"First frame : {frame_ids[0] if frame_ids else None}")
print(f"Last frame  : {frame_ids[-1] if frame_ids else None}")

display(df.head())

===== FILE LOADED =====
Shape       : (11588, 10)
Total frame : 70
First frame : 0
Last frame  : 69


,frame_id,Timestamp,X,Y,Z,X_corr,Y_corr,Z_corr,Z_level,Reflectivity
0,0,437314594380,1.798,-0.981,1.644,2.324326,-0.981,0.730102,1.619816,106.0
1,0,437314594380,1.765,-1.017,1.638,2.291882,-1.017,0.738611,1.628325,28.0
2,0,437314594380,1.800,-1.005,1.541,2.282609,-1.005,0.635907,1.525621,40.0
3,0,437314594380,1.840,-0.970,1.552,2.323510,-0.970,0.628972,1.518686,39.0
4,0,437314594380,1.822,-0.940,1.644,2.346077,-0.940,0.719960,1.609673,26.0


In [14]:
frame_summary_df = (
    df.groupby("frame_id")
    .agg(
        n_points=("frame_id", "count"),
        timestamp_min=("Timestamp", "min"),
        timestamp_max=("Timestamp", "max"),
        x_min=("X_corr", "min"),
        x_max=("X_corr", "max"),
        y_min=("Y_corr", "min"),
        y_max=("Y_corr", "max"),
        z_min=("Z_level", "min"),
        z_max=("Z_level", "max"),
        z_mean=("Z_level", "mean"),
    )
    .reset_index()
)

frame_summary_df["x_range"] = frame_summary_df["x_max"] - frame_summary_df["x_min"]
frame_summary_df["y_range"] = frame_summary_df["y_max"] - frame_summary_df["y_min"]
frame_summary_df["z_range"] = frame_summary_df["z_max"] - frame_summary_df["z_min"]

print("===== FILE SUMMARY =====")
print(f"Total points      : {len(df)}")
print(f"Total frames      : {len(frame_summary_df)}")
print(f"Mean points/frame : {frame_summary_df['n_points'].mean():.2f}")
print(f"Median points     : {frame_summary_df['n_points'].median():.2f}")
print(f"Min points/frame  : {frame_summary_df['n_points'].min()}")
print(f"Max points/frame  : {frame_summary_df['n_points'].max()}")

display(frame_summary_df.head())

===== FILE SUMMARY =====
Total points      : 11588
Total frames      : 70
Mean points/frame : 165.54
Median points     : 155.00
Min points/frame  : 76
Max points/frame  : 279


,frame_id,n_points,timestamp_min,timestamp_max,x_min,x_max,y_min,y_max,z_min,z_max,z_mean,x_range,y_range,z_range
0,0,241,437314594380,437391394380,2.255954,2.554102,-1.301,-0.745,0.087105,1.628325,0.907018,0.298148,0.556,1.541220
1,1,242,437413954380,437491234380,2.252451,2.516038,-1.286,-0.756,0.090711,1.684648,0.902811,0.263587,0.530,1.593937
2,2,250,437496514380,437595874380,2.259330,2.481165,-1.261,-0.737,0.087547,1.670693,0.930850,0.221835,0.524,1.583146
3,3,237,437618434380,437695714380,2.273274,2.523889,-1.263,-0.759,0.088480,1.647080,0.890031,0.250615,0.504,1.558600
4,4,230,437700994380,437795074380,2.273536,2.480264,-1.267,-0.751,0.083173,1.691906,0.890170,0.206728,0.516,1.608733


In [15]:
ROI_X_MIN, ROI_X_MAX = 0.0, 3.0
ROI_Y_MIN, ROI_Y_MAX = -1.5, 1.5
ROI_Z_MIN, ROI_Z_MAX = 0.0, 2.0

MAX_POINTS_PLOT = 15000


def make_frame_figure(frame_id):
    frame_df = df[df["frame_id"] == frame_id].copy()

    if len(frame_df) > MAX_POINTS_PLOT:
        frame_df = frame_df.sample(MAX_POINTS_PLOT, random_state=42)

    fig = go.Figure()

    if len(frame_df) > 0:
        fig.add_trace(
            go.Scatter3d(
                x=frame_df["X_corr"],
                y=frame_df["Y_corr"],
                z=frame_df["Z_level"],
                mode="markers",
                marker=dict(
                    size=2,
                    color=frame_df["Z_level"],
                    colorscale="Turbo",
                    opacity=0.85,
                    colorbar=dict(title="Z_level"),
                ),
                text=[
                    f"frame_id={fid}<br>"
                    f"X_corr={x:.3f}<br>"
                    f"Y_corr={y:.3f}<br>"
                    f"Z_level={z:.3f}<br>"
                    f"Reflectivity={r}"
                    for fid, x, y, z, r in zip(
                        frame_df["frame_id"],
                        frame_df["X_corr"],
                        frame_df["Y_corr"],
                        frame_df["Z_level"],
                        frame_df["Reflectivity"],
                    )
                ],
                hoverinfo="text",
                name="Main cluster",
            )
        )

    fig.update_layout(
        title=(
            f"DBSCAN Main Cluster Visualization<br>"
            f"Frame ID: {frame_id} | Points: {len(frame_df)}"
        ),
        scene=dict(
            xaxis=dict(title="X_corr", range=[ROI_X_MIN, ROI_X_MAX]),
            yaxis=dict(title="Y_corr", range=[ROI_Y_MIN, ROI_Y_MAX]),
            zaxis=dict(title="Z_level", range=[ROI_Z_MIN, ROI_Z_MAX]),
            aspectmode="manual",
            aspectratio=dict(x=1.5, y=1.5, z=1.0),
        ),
        width=950,
        height=750,
        margin=dict(l=0, r=0, b=0, t=90),
    )

    return fig

In [16]:
def make_summary_html(frame_id):
    row = frame_summary_df[frame_summary_df["frame_id"] == frame_id].iloc[0]

    rows = [
        ("frame_id", frame_id),
        ("n_points", row["n_points"]),
        ("timestamp_min", row["timestamp_min"]),
        ("timestamp_max", row["timestamp_max"]),
        ("x_range", row["x_range"]),
        ("y_range", row["y_range"]),
        ("z_range", row["z_range"]),
        ("z_min", row["z_min"]),
        ("z_max", row["z_max"]),
        ("z_mean", row["z_mean"]),
    ]

    html = """
    <div style="font-family: Arial; font-size: 13px;">
    <h3>Frame Summary - Main Cluster Only</h3>
    <table style="border-collapse: collapse;">
    """

    for key, value in rows:
        if isinstance(value, float):
            value_str = f"{value:.6f}"
        else:
            value_str = str(value)

        html += f"""
        <tr>
            <td style="border: 1px solid #ccc; padding: 4px 8px;"><b>{key}</b></td>
            <td style="border: 1px solid #ccc; padding: 4px 8px;">{value_str}</td>
        </tr>
        """

    html += """
    </table>
    </div>
    """

    return HTML(html)

In [17]:
if len(frame_ids) == 0:
    raise ValueError("Tidak ada frame untuk divisualisasikan.")

frame_index_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(frame_ids) - 1,
    step=1,
    description="Frame idx:",
    continuous_update=False,
    layout=widgets.Layout(width="600px"),
)

play_widget = widgets.Play(
    value=0,
    min=0,
    max=len(frame_ids) - 1,
    step=1,
    interval=500,
    description="Play",
    disabled=False,
)

widgets.jslink((play_widget, "value"), (frame_index_slider, "value"))

prev_button = widgets.Button(
    description="Prev Frame",
    tooltip="Go to previous frame",
    icon="arrow-left",
)

next_button = widgets.Button(
    description="Next Frame",
    tooltip="Go to next frame",
    icon="arrow-right",
)

frame_label = widgets.HTML()

plot_output = widgets.Output()
summary_output = widgets.Output()


def update_frame_label():
    idx = frame_index_slider.value
    frame_id = frame_ids[idx]
    frame_label.value = (
        f"<b>Current:</b> index {idx + 1}/{len(frame_ids)} | "
        f"<b>frame_id:</b> {frame_id}"
    )


def render_current_frame(change=None):
    idx = frame_index_slider.value
    frame_id = frame_ids[idx]

    update_frame_label()

    with plot_output:
        clear_output(wait=True)
        fig = make_frame_figure(frame_id)
        fig.show()

    with summary_output:
        clear_output(wait=True)
        display(make_summary_html(frame_id))


def on_prev_clicked(button):
    current = frame_index_slider.value
    if current > frame_index_slider.min:
        frame_index_slider.value = current - 1


def on_next_clicked(button):
    current = frame_index_slider.value
    if current < frame_index_slider.max:
        frame_index_slider.value = current + 1


prev_button.on_click(on_prev_clicked)
next_button.on_click(on_next_clicked)

frame_index_slider.observe(render_current_frame, names="value")

controls = widgets.VBox([
    widgets.HBox([play_widget, frame_index_slider]),
    widgets.HBox([prev_button, next_button]),
    frame_label,
])

display(controls)
display(summary_output)
display(plot_output)

render_current_frame()

Output()

Output()

In [18]:
print("===== FRAMES WITH LOWEST POINT COUNTS =====")
display(
    frame_summary_df
    .sort_values("n_points", ascending=True)
    .head(15)
)

print("===== FRAMES WITH HIGHEST POINT COUNTS =====")
display(
    frame_summary_df
    .sort_values("n_points", ascending=False)
    .head(15)
)

===== FRAMES WITH LOWEST POINT COUNTS =====


,frame_id,n_points,timestamp_min,timestamp_max,x_min,x_max,y_min,y_max,z_min,z_max,z_mean,x_range,y_range,z_range
54,54,76,442689634380,442740514380,1.910795,2.637544,-1.176,-0.753,0.629309,1.134012,0.930414,0.726749,0.423,0.504703
30,30,79,440341474380,440387074380,1.846252,2.606786,-1.155,-0.777,0.792387,1.084152,0.925594,0.760534,0.378,0.291765
41,41,79,441408994380,441463714380,2.367310,2.622015,-1.182,-0.795,0.080184,0.910359,0.458173,0.254705,0.387,0.830175
29,29,99,440236834380,440287714380,1.848421,2.622586,-1.207,-0.776,0.577074,1.073081,0.803749,0.774165,0.431,0.496007
53,53,102,442630114380,442684834380,2.159199,2.770652,-1.191,-0.781,0.082997,0.942324,0.497869,0.611453,0.410,0.859327
31,31,103,440392354380,440441314380,1.973628,2.628818,-1.211,-0.794,0.081584,0.832427,0.538042,0.655190,0.417,0.750843
33,33,122,440591554380,440690914380,1.960090,2.684452,-1.201,-0.789,0.091822,1.080367,0.639235,0.724362,0.412,0.988545
28,28,122,440131714380,440187874380,1.841410,2.609922,-1.175,-0.778,0.380558,1.089787,0.764322,0.768512,0.397,0.709229
52,52,133,442530274380,442584994380,1.914425,2.720084,-1.251,-0.765,0.081725,1.030778,0.636113,0.805659,0.486,0.949053
27,27,140,440027074380,440088514380,1.844007,2.571125,-1.205,-0.780,0.194387,1.084994,0.674783,0.727118,0.425,0.890607


===== FRAMES WITH HIGHEST POINT COUNTS =====


,frame_id,n_points,timestamp_min,timestamp_max,x_min,x_max,y_min,y_max,z_min,z_max,z_mean,x_range,y_range,z_range
69,69,279,444209314380,444276514380,2.094631,2.520015,-1.171,-0.544,0.083883,1.649030,0.938114,0.425384,0.627,1.565147
68,68,251,444104674380,444177154380,2.131551,2.482968,-1.131,-0.588,0.086680,1.627543,0.929403,0.351417,0.543,1.540863
2,2,250,437496514380,437595874380,2.259330,2.481165,-1.261,-0.737,0.087547,1.670693,0.930850,0.221835,0.524,1.583146
1,1,242,437413954380,437491234380,2.252451,2.516038,-1.286,-0.756,0.090711,1.684648,0.902811,0.263587,0.530,1.593937
0,0,241,437314594380,437391394380,2.255954,2.554102,-1.301,-0.745,0.087105,1.628325,0.907018,0.298148,0.556,1.541220
3,3,237,437618434380,437695714380,2.273274,2.523889,-1.263,-0.759,0.088480,1.647080,0.890031,0.250615,0.504,1.558600
67,67,235,444004834380,444072034380,2.117898,2.541832,-1.185,-0.602,0.091339,1.547334,0.889130,0.423934,0.583,1.455995
4,4,230,437700994380,437795074380,2.273536,2.480264,-1.267,-0.751,0.083173,1.691906,0.890170,0.206728,0.516,1.608733
7,7,222,437999074380,438093154380,2.185568,2.506486,-1.261,-0.757,0.086337,1.664077,0.914210,0.320918,0.504,1.577740
6,6,221,437899714380,437993794380,2.239829,2.501031,-1.257,-0.749,0.080262,1.686784,0.913414,0.261202,0.508,1.606522


In [19]:
def jump_to_frame_id(target_frame_id):
    if target_frame_id not in frame_ids:
        print(f"Frame ID {target_frame_id} tidak ditemukan.")
        print(f"Available range: {frame_ids[0]} sampai {frame_ids[-1]}")
        return

    idx = frame_ids.index(target_frame_id)
    frame_index_slider.value = idx
    print(f"Jumped to frame_id={target_frame_id}, index={idx}")


# Contoh:
# jump_to_frame_id(31)